# Markov Decision Processes (MDPs)

## Learning Objectives
- Define the MDP tuple $(S, A, \{P_{sa}\}, \gamma, R)$ and its components
- Understand the **discount factor** $\gamma$ and why it bounds the total payoff
- Define the **value function** $V^\pi(s)$ and derive the **Bellman equation** for a policy
- Define the **optimal value function** $V^*(s)$ and the **Bellman optimality equation**
- Recover the **optimal policy** $\pi^*(s) = \arg\max_a \sum_{s'} P_{sa}(s') V^*(s')$

## Problem Statement

A **Markov Decision Process** is a tuple $(S, A, \{P_{sa}\}, \gamma, R)$:
- $S$: set of **states**
- $A$: set of **actions**
- $P_{sa}$: **transition distribution** — for state $s$, action $a$, gives distribution over next states
- $\gamma \in [0,1)$: **discount factor**
- $R: S \to \mathbb{R}$: **reward function** (immediate payoff)

The agent executes actions $a_t = \pi(s_t)$ and accumulates **discounted total payoff**:
$$R(s_0) + \gamma R(s_1) + \gamma^2 R(s_2) + \cdots$$

**Value function** of policy $\pi$:
$$V^\pi(s) = E\bigl[R(s_0) + \gamma R(s_1) + \gamma^2 R(s_2) + \cdots \mid s_0 = s,\, \pi\bigr]$$

**Bellman equation:**
$$V^\pi(s) = R(s) + \gamma \sum_{s' \in S} P_{s,\pi(s)}(s')\, V^\pi(s')$$

**Optimal value function:**
$$V^*(s) = \max_\pi V^\pi(s)$$

**Bellman optimality equation:**
$$V^*(s) = R(s) + \max_{a \in A}\; \gamma \sum_{s'} P_{sa}(s')\, V^*(s')$$

**Optimal policy:**
$$\pi^*(s) = \arg\max_{a \in A} \sum_{s'} P_{sa}(s')\, V^*(s')$$

## 1. MDP Components and the Discount Factor

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simple grid world: 4x4 grid, absorbing goal state
# States: 0..15, goal at state 15 (bottom-right)
# Actions: 0=up, 1=down, 2=left, 3=right
grid_size = 4
n_states  = grid_size ** 2
n_actions = 4
goal_state = n_states - 1

def state_to_rc(s, g=grid_size):
    return s // g, s % g

def rc_to_state(r, c, g=grid_size):
    return r * g + c

def build_transitions(g=grid_size):
    n = g * g
    # P[s,a,s'] = probability of s' from (s,a)
    P = np.zeros((n, 4, n))
    deltas = [(-1,0), (1,0), (0,-1), (0,1)]  # up, down, left, right
    for s in range(n):
        r, c = state_to_rc(s, g)
        for a, (dr, dc) in enumerate(deltas):
            nr, nc = max(0, min(g-1, r+dr)), max(0, min(g-1, c+dc))
            ns = rc_to_state(nr, nc, g)
            P[s, a, ns] = 1.0
    return P

P = build_transitions()
R = np.full(n_states, -1.0)   # -1 per step
R[goal_state] = 10.0           # reward at goal

# Show the grid world
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
reward_grid = R.reshape(grid_size, grid_size)
im = ax.imshow(reward_grid, cmap='RdYlGn', vmin=-1, vmax=10)
plt.colorbar(im, ax=ax, label='R(s)')
for s in range(n_states):
    r, c = state_to_rc(s)
    ax.text(c, r, f'{s}\nR={R[s]:.0f}', ha='center', va='center', fontsize=7.5)
ax.set_title(f'{grid_size}×{grid_size} Grid World MDP\nGoal = state {goal_state} (R=+10, others R=-1)')
ax.set_xticks([]); ax.set_yticks([])

# Show effect of discount factor on total payoff
ax = axes[1]
T = 30   # horizon
r_per_step = 1.0   # constant reward
t = np.arange(T)
for gamma, color in [(0.5, '#2166ac'), (0.9, '#d6604d'), (0.99, '#4dac26'), (1.0, 'k')]:
    discounts   = gamma ** t
    cum_payoff  = np.cumsum(discounts * r_per_step)
    lbl = f'γ={gamma}  (converges to {r_per_step/(1-gamma):.1f})' if gamma < 1 else 'γ=1 (diverges)'
    ax.plot(t, cum_payoff, color=color, lw=2.5, label=lbl)

ax.set_xlabel('Time step'); ax.set_ylabel('Cumulative discounted payoff')
ax.set_title('Effect of discount factor $\\gamma$\n$\\gamma < 1$ ensures bounded total payoff')
ax.legend(fontsize=8.5)

fig.suptitle('MDP Components: Grid World and Discount Factor', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Bellman Equations and Solving for $V^\pi$

For a fixed policy $\pi$, the Bellman equations form a **linear system** in the values $V^\pi(s)$:
$$V^\pi(s) = R(s) + \gamma \sum_{s'} P_{s,\pi(s)}(s')\, V^\pi(s')$$

In matrix form: $V = R + \gamma P_\pi V$ $\implies$ $V = (I - \gamma P_\pi)^{-1} R$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def policy_value(P, R, policy, gamma):
    """Solve V^pi = (I - gamma * P_pi)^{-1} R."""
    n = len(R)
    P_pi = np.array([P[s, policy[s], :] for s in range(n)])  # (n, n)
    A_mat = np.eye(n) - gamma * P_pi
    return np.linalg.solve(A_mat, R)

gamma = 0.9

# Three policies to compare
policy_rand  = np.random.randint(0, n_actions, n_states)  # random
policy_right = np.full(n_states, 3)                        # always right
# Greedy: always move toward goal (down or right)
policy_good  = np.zeros(n_states, dtype=int)
for s in range(n_states):
    r, c = state_to_rc(s)
    if r < grid_size - 1:
        policy_good[s] = 1   # down
    else:
        policy_good[s] = 3   # right

policies = [('Random policy', policy_rand), ('Always right', policy_right),
            ('Greedy toward goal', policy_good)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
action_arrows = ['↑', '↓', '←', '→']

for ax, (title, pol) in zip(axes, policies):
    V = policy_value(P, R, pol, gamma)
    V_grid = V.reshape(grid_size, grid_size)
    im = ax.imshow(V_grid, cmap='viridis')
    plt.colorbar(im, ax=ax, label='$V^\\pi(s)$')
    for s in range(n_states):
        r, c = state_to_rc(s)
        ax.text(c, r, f'{action_arrows[pol[s]]}\n{V[s]:.1f}',
                ha='center', va='center', fontsize=8,
                color='white' if V[s] < V.mean() else 'black')
    ax.set_title(f'{title}\n$V^\\pi$: white=low, dark=high', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Bellman Equation: $V^\\pi = (I - \\gamma P_\\pi)^{-1}R$ for Three Policies',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Optimal Value Function and Optimal Policy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def value_iteration(P, R, gamma, max_iters=1000, tol=1e-8):
    """Compute V* and pi* via value iteration."""
    n = len(R)
    V = np.zeros(n)
    for _ in range(max_iters):
        Q = R[:, None] + gamma * (P * V[None, None, :]).sum(axis=2)  # (n, |A|)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    pi_star = Q.argmax(axis=1)
    return V, pi_star

V_star, pi_star = value_iteration(P, R, gamma=0.9)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# Optimal value function
ax = axes[0]
V_grid = V_star.reshape(grid_size, grid_size)
im = ax.imshow(V_grid, cmap='YlGn')
plt.colorbar(im, ax=ax, label='$V^*(s)$')
for s in range(n_states):
    r, c = state_to_rc(s)
    ax.text(c, r, f'{V_star[s]:.1f}', ha='center', va='center', fontsize=9)
ax.set_title('Optimal value function $V^*(s)$\n'
             'Higher = closer to goal', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

# Optimal policy
ax = axes[1]
ax.imshow(V_grid, cmap='YlGn', alpha=0.4)
action_arrows = ['↑', '↓', '←', '→']
action_deltas = [(-0.3, 0), (0.3, 0), (0, -0.3), (0, 0.3)]
for s in range(n_states):
    if s == goal_state:
        continue
    r, c = state_to_rc(s)
    a = pi_star[s]
    dr, dc = action_deltas[a]
    ax.annotate('', xy=(c+dc, r+dr), xytext=(c, r),
                arrowprops=dict(arrowstyle='->', color='#2166ac', lw=2, mutation_scale=15))
ax.scatter(*state_to_rc(goal_state)[::-1], s=400, c='gold', marker='*', zorder=5)
ax.set_title('Optimal policy $\\pi^*(s)$\n★ = goal state', fontsize=10)
ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('MDP Solution: Optimal $V^*$ and $\\pi^*$  (γ=0.9)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Optimal policy actions:', [action_arrows[a] for a in pi_star])
print('Optimal value at start state (s=0):', V_star[0].round(3))

## 4. Stochastic Transitions and the Effect of $\gamma$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def build_stochastic_transitions(g=grid_size, slip=0.1):
    """With probability slip, a random action is taken instead."""
    P_det = build_transitions(g)
    n = g * g
    P_stoch = (1 - slip) * P_det + slip * P_det.mean(axis=1, keepdims=True)
    # Re-normalise
    P_stoch /= P_stoch.sum(axis=2, keepdims=True)
    return P_stoch

P_stoch = build_stochastic_transitions(slip=0.2)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, (P_use, ptitle) in enumerate([(P, 'Deterministic'), (P_stoch, 'Stochastic (20% slip)')]):
    for col, gamma_val in enumerate([0.5, 0.9, 0.99]):
        V_g, pi_g = value_iteration(P_use, R, gamma_val)
        ax = axes[row, col]
        im = ax.imshow(V_g.reshape(grid_size, grid_size), cmap='YlGn')
        for s in range(n_states):
            r, c = state_to_rc(s)
            ax.text(c, r, f'{action_arrows[pi_g[s]]}\n{V_g[s]:.1f}',
                    ha='center', va='center', fontsize=7.5)
        ax.set_title(f'{ptitle}, γ={gamma_val}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Optimal Policy: Effect of Discount Factor and Stochastic Transitions',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">MDP tuple</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">(S,A,P&#x209b;&#x2090;,&#x3b3;,R)</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >states S, actions A, transitions P&#x209b;&#x2090;(s&#x2032;), discount &#x3b3;, rewards R(s)</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >define policy &#x3c0;: S&#x2192;A and value function</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Policy &#x3c0;</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">V&#x3c0;(s)</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >V&#x3c0;(s) = E[&#x2211;&#x1d57; &#x3b3;&#x1d57; R(s&#x1d57;) | s&#x2080;=s, &#x3c0;] &#x2014; expected discounted return</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >write Bellman equation for V&#x3c0;</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Bellman eq.</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">(policy)</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >V&#x3c0;(s) = R(s) + &#x3b3; &#x2211;&#x209b;&#x2019; P&#x209b;&#x3c0;&#x208d;&#x209b;&#x208e;(s&#x2019;) V&#x3c0;(s&#x2019;) &#x2014; linear system, solvable</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >find optimal policy V*</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Optimal V*</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >V*(s) = max&#x2090; [R(s) + &#x3b3; &#x2211;&#x209b;&#x2019; P&#x209b;&#x2090;(s&#x2019;) V*(s&#x2019;)] &#x2014; Bellman optimality</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| MDP | $(S, A, P_{sa}, \gamma, R)$ | Formalises sequential decision-making |
| Value function | $V^\pi(s) = E\left[\sum_t \gamma^t R(s_t)\mid s_0=s, \pi\right]$ | Expected discounted return under policy $\pi$ |
| Bellman equation (policy) | $V^\pi(s) = R(s) + \gamma \sum_{s'} P_{s\pi(s)}(s') V^\pi(s')$ | Linear system — solvable for any fixed $\pi$ |
| Optimal value | $V^*(s) = \max_a\left[R(s) + \gamma\sum_{s'} P_{sa}(s')V^*(s')\right]$ | Bellman optimality equation |
| Optimal policy | $\pi^*(s) = \arg\max_a\sum_{s'} P_{sa}(s')V^*(s')$ | Greedy w.r.t. $V^*$ |
| Discount $\gamma$ | $0 \leq \gamma < 1$ | Ensures convergence; trades off immediate vs future reward |

**Key insight:** the Bellman equation recursively defines the value function in terms of immediate reward plus discounted future value; the optimal value function $V^*$ satisfies the Bellman optimality equation and is found by value or policy iteration.